# Sensor Modality Decision Contribution

The original section analyzes the standardized saved models using all available windows. The ACC-excluded section retrains each model after removing every ACC-derived feature, then analyzes the remaining modalities on the same windows.

In [13]:
from pathlib import Path
import re
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.svm import SVC
from xgboost import XGBClassifier

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == '4th try':
    REPO_ROOT = REPO_ROOT.parents[1]
elif not (REPO_ROOT / 'Models').exists() and (Path.cwd().parents[1] / 'Models').exists():
    REPO_ROOT = Path.cwd().parents[1]

DATASET_DIR = REPO_ROOT / 'Data' / 'Preprocessed n STAI-labeled 6s'
MODEL_DIR = REPO_ROOT / 'Models' / 'stai_6s'
OUTPUT_DIR = REPO_ROOT / 'Data' / 'Analysis' / 'sensor_modality_decision_contribution'
ACC_EXCLUDED_OUTPUT_DIR = REPO_ROOT / 'Data' / 'Analysis_old' / 'sensor_modality_decision_contribution_acc_excluded'
ACC_EXCLUDED_MODEL_DIR = REPO_ROOT / 'Models_old' / 'stai_6s_standardized_acc_excluded'

METADATA_COLUMNS = {'binary_anxiety', 'subject_id', 'condition', 'window_start_seconds', 'window_end_seconds', 'condition_stai_mean', 'baseline_stai_mean'}
MODALITY_PREFIXES = ('net_acc', 'ACC', 'BVP', 'EDA', 'TEMP', 'Temp', 'ECG', 'EMG', 'Resp')
MODEL_FILES = {'Random Forest': 'stai_baseline_relative_random_forest_model.joblib', 'XGBoost': 'stai_baseline_relative_xgboost_model.joblib', 'SVM': 'stai_baseline_relative_svm_model.joblib', 'KNN': 'stai_baseline_relative_knn_model.joblib'}


In [14]:
def load_all_windows(dataset_dir):
    paths = sorted(dataset_dir.glob('S*_stai_60s_shift6s.pkl'))
    if not paths:
        raise FileNotFoundError(f'No STAI 6-second datasets found in {dataset_dir}')
    data = pd.concat([pd.read_pickle(path) for path in paths], ignore_index=True)
    data = data[data['binary_anxiety'].isin([0, 1])].copy()
    feature_columns = [c for c in data.columns if c not in METADATA_COLUMNS]
    data[feature_columns] = data[feature_columns].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0)
    print(f'Analyzing {len(data)} identical windows from {len(paths)} subject files')
    print('Labels:', data['binary_anxiety'].value_counts().sort_index().to_dict())
    return data, feature_columns

def modality_for_feature(feature_name):
    name = re.sub(r'^ACC_sub_', 'ACC_', feature_name)
    for prefix in MODALITY_PREFIXES:
        if name == prefix or name.startswith(f'{prefix}_'):
            return 'ACC' if prefix == 'net_acc' else prefix.upper()
    return 'Other'

def decision_score(model, X):
    if hasattr(model, 'predict_proba'):
        probabilities = model.predict_proba(X)
        classes = list(getattr(model, 'classes_', [0, 1]))
        return probabilities[:, classes.index(1)] if 1 in classes else np.zeros(len(X))
    raw = np.clip(np.asarray(model.decision_function(X)).reshape(-1), -50, 50)
    return 1 / (1 + np.exp(-raw))

WINDOWS, FEATURE_COLUMNS = load_all_windows(DATASET_DIR)
X_ALL = WINDOWS[FEATURE_COLUMNS]
Y_ALL = WINDOWS['binary_anxiety'].astype(int)
FEATURE_MODALITIES = pd.Series([modality_for_feature(c) for c in FEATURE_COLUMNS], index=FEATURE_COLUMNS)


Analyzing 4147 identical windows from 15 subject files
Labels: {0: 3404, 1: 743}


## Original standardized models

Each model below uses all available windows and the standardized model saved by the training notebook.

In [15]:
def analyze_decision_contribution(model_name, filename):
    model = joblib.load(MODEL_DIR / filename)
    expected = getattr(model, 'n_features_in_', None)
    if expected is not None and expected != len(FEATURE_COLUMNS):
        raise ValueError(f'{model_name} expects {expected} features, but dataset provides {len(FEATURE_COLUMNS)}.')
    original_pred = np.asarray(model.predict(X_ALL)).astype(int)
    original_score = decision_score(model, X_ALL)
    medians = X_ALL.median()
    rows = []
    for modality in sorted(FEATURE_MODALITIES.unique()):
        columns = FEATURE_MODALITIES[FEATURE_MODALITIES == modality].index.tolist()
        masked = X_ALL.copy()
        masked[columns] = medians[columns].to_numpy()
        masked_pred = np.asarray(model.predict(masked)).astype(int)
        score_change = np.abs(original_score - decision_score(model, masked))
        pred_change = (original_pred != masked_pred).astype(int)
        for label in [0, 1]:
            selected = original_pred == label
            rows.append({'modality': modality, 'predicted_label': label, 'windows': int(selected.sum()), 'mean_absolute_decision_change': float(score_change[selected].mean()) if selected.any() else 0.0, 'prediction_change_rate': float(pred_change[selected].mean()) if selected.any() else 0.0})
    detail = pd.DataFrame(rows)
    detail['weighted_score_change'] = detail['mean_absolute_decision_change'] * detail['windows']
    detail['weighted_prediction_change'] = detail['prediction_change_rate'] * detail['windows']
    overall = detail.groupby('modality', as_index=False).agg(windows=('windows', 'sum'), weighted_score_change=('weighted_score_change', 'sum'), weighted_prediction_change=('weighted_prediction_change', 'sum'))
    overall['mean_absolute_decision_change'] = overall['weighted_score_change'] / overall['windows']
    overall['prediction_change_rate'] = overall['weighted_prediction_change'] / overall['windows']
    overall = overall.drop(columns=['weighted_score_change', 'weighted_prediction_change']).sort_values('mean_absolute_decision_change', ascending=False, ignore_index=True)
    total = overall['mean_absolute_decision_change'].sum()
    overall['relative_contribution'] = overall['mean_absolute_decision_change'] / total if total else 0.0
    overall['rank'] = np.arange(1, len(overall) + 1)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    stem = Path(filename).stem
    overall.to_csv(OUTPUT_DIR / f'{stem}_overall.csv', index=False)
    detail.to_csv(OUTPUT_DIR / f'{stem}_by_predicted_label.csv', index=False)

    plot_data = overall.sort_values('mean_absolute_decision_change')
    values = plot_data['relative_contribution'] * 100
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(plot_data['modality'], values)
    ax.set_xlabel('Relative contribution to decision change (%)')
    ax.set_ylabel('Sensor modality')
    ax.set_title(f'{model_name}: modality decision contribution')
    for index, value in enumerate(values):
        ax.text(value, index, f' {value:.2f}%', va='center')
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f'{stem}_overall.png', dpi=160)
    plt.show()
    plt.close(fig)

    print(f'{model_name}: {len(X_ALL)} identical windows; top modality: {overall.iloc[0]["modality"]} ({overall.iloc[0]["relative_contribution"]:.2%})')
    display(overall)
    display(detail)
    return overall, detail


### Random Forest

In [16]:
random_forest_overall, random_forest_by_label = analyze_decision_contribution('Random Forest', MODEL_FILES['Random Forest'])


FileNotFoundError: [Errno 2] No such file or directory: 'e:\\Rizqy\\Kuliah\\Skripsi\\repo-skripsi\\Models\\stai_6s\\stai_baseline_relative_random_forest_model.joblib'

### XGBoost

In [ ]:
xgboost_overall, xgboost_by_label = analyze_decision_contribution('XGBoost', MODEL_FILES['XGBoost'])


### SVM

In [ ]:
svm_overall, svm_by_label = analyze_decision_contribution('SVM', MODEL_FILES['SVM'])


### KNN

In [ ]:
knn_overall, knn_by_label = analyze_decision_contribution('KNN', MODEL_FILES['KNN'])


## ACC-excluded models

These models are retrained using the same complete windows after removing all ACC-derived features. They are separate models because the original standardized models require the original feature count.

In [ ]:
NON_ACC_COLUMNS = [column for column in FEATURE_COLUMNS if FEATURE_MODALITIES[column] != 'ACC']
X_NO_ACC = X_ALL[NON_ACC_COLUMNS].copy()
FEATURE_MODALITIES_NO_ACC = FEATURE_MODALITIES[NON_ACC_COLUMNS]
ACC_EXCLUDED_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ACC_EXCLUDED_MODEL_DIR.mkdir(parents=True, exist_ok=True)
print(f'ACC-excluded analysis: {len(X_NO_ACC)} windows, {len(NON_ACC_COLUMNS)} non-ACC features')
print(f'Removed {len(FEATURE_COLUMNS) - len(NON_ACC_COLUMNS)} ACC features')

def scaled_estimator(estimator):
    return Pipeline([('standardize', StandardScaler()), ('normalize', MinMaxScaler()), ('model', estimator)])

def analyze_acc_excluded_model(model_name, model):
    model.fit(X_NO_ACC, Y_ALL)
    original_pred = np.asarray(model.predict(X_NO_ACC)).astype(int)
    original_score = decision_score(model, X_NO_ACC)
    medians = X_NO_ACC.median()
    rows = []
    for modality in sorted(FEATURE_MODALITIES_NO_ACC.unique()):
        columns = FEATURE_MODALITIES_NO_ACC[FEATURE_MODALITIES_NO_ACC == modality].index.tolist()
        masked = X_NO_ACC.copy()
        masked[columns] = medians[columns].to_numpy()
        masked_pred = np.asarray(model.predict(masked)).astype(int)
        score_change = np.abs(original_score - decision_score(model, masked))
        pred_change = (original_pred != masked_pred).astype(int)
        for label in [0, 1]:
            selected = original_pred == label
            rows.append({'modality': modality, 'predicted_label': label, 'windows': int(selected.sum()), 'mean_absolute_decision_change': float(score_change[selected].mean()) if selected.any() else 0.0, 'prediction_change_rate': float(pred_change[selected].mean()) if selected.any() else 0.0})
    detail = pd.DataFrame(rows)
    detail['weighted_score_change'] = detail['mean_absolute_decision_change'] * detail['windows']
    detail['weighted_prediction_change'] = detail['prediction_change_rate'] * detail['windows']
    overall = detail.groupby('modality', as_index=False).agg(windows=('windows', 'sum'), weighted_score_change=('weighted_score_change', 'sum'), weighted_prediction_change=('weighted_prediction_change', 'sum'))
    overall['mean_absolute_decision_change'] = overall['weighted_score_change'] / overall['windows']
    overall['prediction_change_rate'] = overall['weighted_prediction_change'] / overall['windows']
    overall = overall.drop(columns=['weighted_score_change', 'weighted_prediction_change']).sort_values('mean_absolute_decision_change', ascending=False, ignore_index=True)
    total = overall['mean_absolute_decision_change'].sum()
    overall['relative_contribution'] = overall['mean_absolute_decision_change'] / total if total else 0.0
    overall['rank'] = np.arange(1, len(overall) + 1)
    safe_name = model_name.lower().replace(' ', '_')
    joblib.dump(model, ACC_EXCLUDED_MODEL_DIR / f'{safe_name}_acc_excluded.joblib')
    overall.to_csv(ACC_EXCLUDED_OUTPUT_DIR / f'{safe_name}_overall.csv', index=False)
    detail.to_csv(ACC_EXCLUDED_OUTPUT_DIR / f'{safe_name}_by_predicted_label.csv', index=False)

    plot_data = overall.sort_values('mean_absolute_decision_change')
    values = plot_data['relative_contribution'] * 100
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(plot_data['modality'], values)
    ax.set_xlabel('Relative contribution to decision change (%)')
    ax.set_ylabel('Sensor modality')
    ax.set_title(f'{model_name} (ACC excluded)')
    for index, value in enumerate(values):
        ax.text(value, index, f' {value:.2f}%', va='center')
    fig.tight_layout()
    fig.savefig(ACC_EXCLUDED_OUTPUT_DIR / f'{safe_name}_overall.png', dpi=160)
    plt.show()
    plt.close(fig)

    print(f'{model_name} (ACC excluded): {len(X_NO_ACC)} identical windows; top remaining modality: {overall.iloc[0]["modality"]} ({overall.iloc[0]["relative_contribution"]:.2%})')
    display(overall)
    display(detail)
    return overall, detail

ACC_EXCLUDED_MODELS = {
    'Random Forest': scaled_estimator(RandomForestClassifier(n_estimators=100, random_state=100)),
    'XGBoost': scaled_estimator(XGBClassifier(n_estimators=100, eval_metric='logloss', random_state=100)),
    'SVM': scaled_estimator(SVC(kernel='rbf', random_state=100)),
    'KNN': scaled_estimator(KNeighborsClassifier(n_neighbors=5)),
}


### Random Forest without ACC

In [ ]:
random_forest_acc_excluded_overall, random_forest_acc_excluded_by_label = analyze_acc_excluded_model('Random Forest', ACC_EXCLUDED_MODELS['Random Forest'])


### XGBoost without ACC

In [ ]:
xgboost_acc_excluded_overall, xgboost_acc_excluded_by_label = analyze_acc_excluded_model('XGBoost', ACC_EXCLUDED_MODELS['XGBoost'])


### SVM without ACC

In [ ]:
svm_acc_excluded_overall, svm_acc_excluded_by_label = analyze_acc_excluded_model('SVM', ACC_EXCLUDED_MODELS['SVM'])


### KNN without ACC

In [ ]:
knn_acc_excluded_overall, knn_acc_excluded_by_label = analyze_acc_excluded_model('KNN', ACC_EXCLUDED_MODELS['KNN'])
